# MiniLLM Theory Verification Experiments

Runnable code demos for core LLM concepts.

Dependencies: `pip install torch numpy matplotlib`

In [ ]:
import torch
import torch.nn.functional as F
import math
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(""), ".."))

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Cross-Entropy Loss

Pre-training objective: predict next token.

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(x_t | x_{<t})$$

In [ ]:
# Simulate: 4 tokens, vocab size 10
torch.manual_seed(42)
V, T = 10, 4
logits = torch.randn(T, V)
targets = torch.tensor([3, 7, 1, 5])

# Method 1: manual cross entropy
log_probs = F.log_softmax(logits, dim=-1)
per_token_loss = -log_probs[range(T), targets]
manual_loss = per_token_loss.mean()

# Method 2: PyTorch built-in
pytorch_loss = F.cross_entropy(logits, targets)

print(f"Manual CE loss: {manual_loss:.4f}")
print(f"PyTorch CE loss: {pytorch_loss:.4f}")
print(f"Match: {torch.allclose(manual_loss, pytorch_loss)}")
print(f"\nRandom prediction loss ~= ln({V}) = {math.log(V):.4f}")
print(f"Actual loss = {pytorch_loss:.4f}")

## 2. Causal Mask Verification

Changing future tokens should NOT affect past positions.

In [ ]:
from model.config import MiniLLMConfig
from model.gpt import MiniLLM

config = MiniLLMConfig(n_layer=2, n_head=2, n_embd=64, block_size=128, vocab_size=100)
model = MiniLLM(config)
model.eval()

torch.manual_seed(42)
x = torch.randint(0, 100, (1, 20))

with torch.no_grad():
    out1 = model(x)["logits"]

x_mod = x.clone()
x_mod[0, 10:] = torch.randint(0, 100, (10,))

with torch.no_grad():
    out2 = model(x_mod)["logits"]

diff_before = (out1[0, :10] - out2[0, :10]).abs().max().item()
diff_after = (out1[0, 10:] - out2[0, 10:]).abs().max().item()

print(f"Positions 0-9  max diff:  {diff_before:.8f} (should be ~0)")
print(f"Positions 10-19 max diff: {diff_after:.4f} (should be >0)")
print(f"\nCausal mask: {"VALID!" if diff_before < 1e-5 else "BROKEN!"}")

## 3. DPO Loss

$$\mathcal{L} = -\log \sigma\left(\beta \left[\Delta_{policy} - \Delta_{ref}\right]\right)$$

where $\Delta = \log P(y_w|x) - \log P(y_l|x)$

In [ ]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    policy_ratio = policy_chosen - policy_rejected
    ref_ratio = ref_chosen - ref_rejected
    logits = beta * (policy_ratio - ref_ratio)
    return -F.logsigmoid(logits).mean()

beta = 0.1

# Scenario 1: policy correctly prefers chosen
loss_good = dpo_loss(torch.tensor([0.5]), torch.tensor([-0.5]), torch.tensor([0.0]), torch.tensor([0.0]), beta=beta)

# Scenario 2: policy wrongly prefers rejected
loss_bad = dpo_loss(torch.tensor([-0.5]), torch.tensor([0.5]), torch.tensor([0.0]), torch.tensor([0.0]), beta=beta)

# Scenario 3: policy = reference
loss_neutral = dpo_loss(torch.tensor([0.5]), torch.tensor([-0.5]), torch.tensor([0.5]), torch.tensor([-0.5]), beta=beta)

print(f"Policy prefers chosen (good):    loss = {loss_good:.4f}")
print(f"Policy prefers rejected (bad):    loss = {loss_bad:.4f}")
print(f"Policy = reference (neutral):    loss = {loss_neutral:.4f}")
print(f"\nloss_bad > loss_neutral > loss_good: {loss_bad > loss_neutral > loss_good}")

## 4. Beta Parameter Effect

Beta controls how aggressively DPO adjusts preferences.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

betas = torch.linspace(0.01, 1.0, 100)
margin = 0.5

losses = [-F.logsigmoid(torch.tensor(b * margin)).item() for b in betas]

plt.figure(figsize=(8, 4))
plt.plot(betas.numpy(), losses)
plt.xlabel('beta')
plt.ylabel('DPO Loss')
plt.title('DPO Loss vs Beta (fixed margin=0.5)')
plt.grid(True)
plt.savefig('docs/beta_plot.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved to docs/beta_plot.png')
print(f"beta=0.05 (conservative): loss={-F.logsigmoid(torch.tensor(0.025)):.4f}")
print(f"beta=0.10 (default):      loss={-F.logsigmoid(torch.tensor(0.05)):.4f}")
print(f"beta=0.50 (aggressive):   loss={-F.logsigmoid(torch.tensor(0.25)):.4f}")

## 5. RoPE: Rotary Position Embedding

Verify that dot product depends only on relative position.

In [ ]:
def apply_rope_simple(x, pos, theta=10000.0):
    d = x.shape[-1]
    freqs = 1.0 / (theta ** (torch.arange(0, d, 2).float() / d))
    angles = pos * freqs
    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    r_even = x_even * cos_a - x_odd * sin_a
    r_odd = x_even * sin_a + x_odd * cos_a
    return torch.stack([r_even, r_odd], dim=-1).flatten(-2)

d = 8
q, k = torch.randn(d), torch.randn(d)

print("Positions (m,n) | dot(q_m, k_n) | relative m-n")
print("-" * 50)
for m, n in [(0,0), (0,1), (0,2), (1,3), (2,4)]:
    dot = (apply_rope_simple(q, m) * apply_rope_simple(k, n)).sum().item()
    print(f"({m}, {n})            | {dot:>13.4f}   | {m-n}")

# Same relative position should give same dot product
dot_02 = (apply_rope_simple(q, 0) * apply_rope_simple(k, 2)).sum().item()
dot_24 = (apply_rope_simple(q, 2) * apply_rope_simple(k, 4)).sum().item()
print(f"\n(0,2) dot: {dot_02:.4f}")
print(f"(2,4) dot: {dot_24:.4f}")
print(f"Equal (same relative pos): {abs(dot_02 - dot_24) < 1e-5}")

## 6. Activation Functions: ReLU vs SiLU

$$\text{SiLU}(x) = x \cdot \sigma(x)$$

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

x = torch.linspace(-4, 4, 200)

plt.figure(figsize=(8, 4))
plt.plot(x.numpy(), F.relu(x).numpy(), label='ReLU')
plt.plot(x.numpy(), F.silu(x).numpy(), label='SiLU/Swish')
plt.plot(x.numpy(), F.gelu(x).numpy(), label='GELU')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Activation Functions')
plt.legend()
plt.grid(True)
plt.savefig('docs/activation_plot.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved to docs/activation_plot.png')
print('\nSiLU advantages:')
print('  - Non-zero output for x < 0 (unlike ReLU hard cutoff)')
print('  - Smooth gradient, more stable training')
print('  - Gating mechanism in SwiGLU allows selective information flow')

## 7. Best-of-N Sampling (RSFT Principle)

Generate N candidates, pick the best one.

In [ ]:
def reward(text):
    words = text.split()
    if len(words) < 5: return 0.0
    length = min(len(words) / 30.0, 1.0)
    diversity = len(set(w.lower() for w in words)) / len(words)
    complete = 1.0 if text.strip()[-1] in ".!?" else 0.0
    return length * 0.3 + diversity * 0.4 + complete * 0.3

candidates = [
    "The cat.",
    "The cat sat on the mat and looked at the butterfly.",
    "cat cat cat cat cat",
    "The little dog ran to the park and played with friends.",
    "A bird.",
    "Once upon a time there was a beautiful garden.",
    "the the the the the",
    "She opened the door and saw a big surprise.",
]

scored = sorted([(reward(c), c) for c in candidates], key=lambda x: -x[0])
print("Best-of-N Sampling Demo:")
print("=" * 60)
for i, (s, t) in enumerate(scored):
    tag = " <-- BEST" if i == 0 else ""
    print(f"  {s:.2f} | {t[:55]}{tag}")
print("\nRSFT: train on the best candidates = model improves without leaving its distribution")